# LangGraph 005 — Core Concepts

State, reducers, super-steps and what `compile()` really checks — each one run
rather than described. Needs `langgraph`; **no API key**.

| Part | What we check |
|---|---|
| A | a node sees the whole state and returns only what it changed |
| B | replace keeps **1 of 4** messages; `add` keeps **4** |
| C | two parallel writes to one key raise `InvalidUpdateError` |
| D | the same 4 nodes: **2** super-steps parallel, **4** sequential |
| E | which broken graphs `compile()` actually rejects |
| F | the five workflow patterns, built and counted |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — State

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BlogState(TypedDict):      # the state: the shared memory of the workflow
    topic: str
    outline: str
    report: str

def write_outline(state: BlogState) -> dict:
    print("the node was handed:", state)
    return {"outline": "1. intro  2. body"}    # return ONLY what changed

graph = StateGraph(BlogState)
graph.add_node("write_outline", write_outline)
graph.add_edge(START, "write_outline")
graph.add_edge("write_outline", END)

print(graph.compile().invoke({"topic": "cricket", "outline": "", "report": ""}))
# the node was handed: {'topic': 'cricket', 'outline': '', 'report': ''}
# {'topic': 'cricket', 'outline': '1. intro  2. body', 'report': ''}
#
# Every node sees the whole state. A node returns a partial update, and LangGraph
# merges it: 'topic' and 'report' are untouched, not wiped.

In [ ]:
out = graph.compile().invoke({"topic": "cricket", "outline": "", "report": ""})
assert out["topic"] == "cricket" and out["report"] == ""      # untouched keys survive
assert out["outline"] == "1. intro  2. body"
print("a partial update is merged, not a replacement")

## Part B — Reducers

In [ ]:
import operator
from typing import Annotated

TURNS = ["Hi, my name is Ayesha", "Hello! How can I help?",
         "What is my name?", "Your name is Ayesha"]

class Replace(TypedDict):
    messages: list                               # default: a write replaces

class Append(TypedDict):
    messages: Annotated[list, operator.add]      # reducer: a write appends

def build(State):
    g = StateGraph(State)
    previous = START
    for i, turn in enumerate(TURNS):
        g.add_node(f"turn{i}", lambda s, t=turn: {"messages": [t]})
        g.add_edge(previous, f"turn{i}")
        previous = f"turn{i}"
    g.add_edge(previous, END)
    return g.compile()

for name, State in (("replace", Replace), ("add", Append)):
    final = build(State).invoke({"messages": []})
    print(f"{name:<8} {len(final['messages'])} message(s): {final['messages']}")
# replace  1 message(s): ['Your name is Ayesha']
# add      4 message(s): ['Hi, my name is Ayesha', 'Hello! How can I help?', ...]
#
# With the default, each turn overwrites the last, so by the time the chatbot is
# asked for the name, the turn that gave the name is gone.

In [ ]:
assert len(build(Replace).invoke({"messages": []})["messages"]) == 1
assert len(build(Append).invoke({"messages": []})["messages"]) == 4
print("replace: 1 of 4 | add: 4 of 4")

## Part C — Parallel writes

Two branches, one key. Without a reducer LangGraph refuses to pick a winner.

In [ ]:
# Two parallel nodes writing the SAME key. Without a reducer LangGraph refuses.
def build(annotated: bool):
    class Plain(TypedDict):
        flags: list

    class Reduced(TypedDict):
        flags: Annotated[list, operator.add]

    g = StateGraph(Reduced if annotated else Plain)
    for name in ("check_a", "check_b"):
        g.add_node(name, lambda s, n=name: {"flags": [n]})
        g.add_edge(START, name)       # both start at once
        g.add_edge(name, END)
    return g.compile()

for annotated in (False, True):
    try:
        print("reducer" if annotated else "no reducer", "->", build(annotated).invoke({"flags": []}))
    except Exception as e:
        print("no reducer ->", type(e).__name__, "-", str(e).splitlines()[0])
# no reducer -> InvalidUpdateError - At key 'flags': Can receive only one value per step.
#               Use an Annotated key to handle multiple values
# reducer -> {'flags': ['check_a', 'check_b']}

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    build(False).invoke({"flags": []})
    raise AssertionError("expected InvalidUpdateError")
except InvalidUpdateError:
    print("refused, as it should be")
assert build(True).invoke({"flags": []})["flags"] == ["check_a", "check_b"]

## Part D — Super-steps

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

class S(TypedDict):
    log: Annotated[list, operator.add]

def build(parallel: bool, saver):
    g = StateGraph(S)
    for n in ("a", "b", "c"):
        g.add_node(n, lambda s, n=n: {"log": [n]})
    g.add_node("join", lambda s: {"log": ["join"]})
    if parallel:
        for n in ("a", "b", "c"):
            g.add_edge(START, n)      # a, b and c all leave START together
            g.add_edge(n, "join")
    else:
        g.add_edge(START, "a"); g.add_edge("a", "b")
        g.add_edge("b", "c"); g.add_edge("c", "join")
    g.add_edge("join", END)
    return g.compile(checkpointer=saver)

for parallel in (True, False):
    saver = InMemorySaver()
    cfg = {"configurable": {"thread_id": str(parallel)}}
    app = build(parallel, saver)
    app.invoke({"log": []}, cfg)
    steps = max(h.metadata["step"] for h in app.get_state_history(cfg))
    print(f"{'parallel' if parallel else 'sequential':<11} 4 nodes in {steps} super-steps")
# parallel    4 nodes in 2 super-steps
# sequential  4 nodes in 4 super-steps
#
# stream_mode="updates" will not show you this: it emits one chunk per node either
# way. The checkpointer is what numbers the super-steps.

In [ ]:
def steps_for(parallel):
    saver = InMemorySaver()
    cfg = {"configurable": {"thread_id": f"check-{parallel}"}}
    app = build(parallel, saver)
    app.invoke({"log": []}, cfg)
    return max(h.metadata["step"] for h in app.get_state_history(cfg))

assert steps_for(True) == 2 and steps_for(False) == 4
# the update stream cannot show this: one chunk per node either way
saver = InMemorySaver()
chunks = list(build(True, saver).stream({"log": []}, {"configurable": {"thread_id": "s"}},
                                        stream_mode="updates"))
print(f"parallel: 2 super-steps, but stream_mode='updates' gave {len(chunks)} chunks")

## Part E — What `compile()` catches

In [ ]:
class S(TypedDict):
    x: int

def try_compile(build, label):
    g = StateGraph(S)
    build(g)
    try:
        g.compile()
        print(f"{label:<26} compiles")
    except Exception as e:
        print(f"{label:<26} {type(e).__name__}: {str(e).splitlines()[0]}")

def orphan(g):
    g.add_node("used", lambda s: {"x": 1}); g.add_node("orphan", lambda s: {"x": 2})
    g.add_edge(START, "used"); g.add_edge("used", END)

def ghost_edge(g):
    g.add_node("used", lambda s: {"x": 1})
    g.add_edge(START, "used"); g.add_edge("used", "ghost")

def no_start(g):
    g.add_node("used", lambda s: {"x": 1}); g.add_edge("used", END)

try_compile(orphan, "orphan node, no edges")
try_compile(ghost_edge, "edge to unknown node")
try_compile(no_start, "no edge from START")
# orphan node, no edges      compiles
# edge to unknown node       ValueError: Found edge ending at unknown node `ghost`
# no edge from START         ValueError: Graph must have an entrypoint: ...

In [ ]:
# An orphan node is silent: the graph runs and the node simply never fires.
ran = []
g = StateGraph(S)
g.add_node("used", lambda s: ran.append("used") or {"x": 1})
g.add_node("orphan", lambda s: ran.append("orphan") or {"x": 2})
g.add_edge(START, "used"); g.add_edge("used", END)
g.compile().invoke({"x": 0})
assert ran == ["used"]
print("orphan node never ran, and nothing warned you:", ran)

## Part F — The five patterns

Scripted "model" calls, so the counts are real and the cost is zero.

In [ ]:
CALLS = []
def fake_llm(tag, out):
    CALLS.append(tag)
    return out

# evaluator-optimizer: generate, score, regenerate until it passes
class Draft(TypedDict):
    text: str
    score: int
    rounds: int

scores = iter([4, 6, 9])
g = StateGraph(Draft)
g.add_node("generate", lambda s: {"text": fake_llm("generator", "a draft"), "rounds": s["rounds"] + 1})
g.add_node("evaluate", lambda s: {"score": fake_llm("evaluator", next(scores))})
g.add_edge(START, "generate"); g.add_edge("generate", "evaluate")
g.add_conditional_edges("evaluate", lambda s: END if s["score"] >= 8 else "generate")

out = g.compile().invoke({"text": "", "score": 0, "rounds": 0})
print(f"accepted at {out['score']} after {out['rounds']} drafts, {len(CALLS)} model calls")
assert out["rounds"] == 3 and len(CALLS) == 6

## What to take away

- A node receives the **whole** state and returns **only what it changed**.
- The default write **replaces**. Conversations need `Annotated[list, add]`.
- Parallel branches writing one key need a reducer, or you get a loud error.
- Progress is counted in **super-steps**: 2 parallel against 4 sequential here.
- `compile()` does **not** catch an orphan node. Check your edges.

## Exercises

1. Put an `add` reducer on the essay key of a rewrite loop so every draft is kept.
   How many drafts after three rewrites?
2. Give the evaluator a pass mark of 10 and a scripted score list that never
   reaches it. What stops the loop? (Look up `recursion_limit` from Generative AI 020.)
3. Build the routing pattern: one classifier node and three specialists, and
   confirm only one specialist runs.